<a href="https://colab.research.google.com/github/23MH1A05M8/-Product-Review-Event-Processor-with-Sentiment-Analysis-and-Message-Queues/blob/main/Day2_Setup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [33]:
!pip install -q google-genai pydantic

import os
import getpass

if 'GEMINI_API_KEY' not in os.environ:
    os.environ['GEMINI_API_KEY'] = getpass.getpass("Gemini API Key: ")

In [34]:
from pydantic import BaseModel
from typing import List, Optional

class Education(BaseModel):
    degree: str
    institution: str
    year: int

class Resume(BaseModel):
    name: str
    email: str
    phone: Optional[str] = None
    education: List[Education]
    skills: List[str]
    projects: List[str] = []
    experience_years: float

In [35]:
from google import genai
from pydantic import ValidationError

client = genai.Client(api_key=os.environ['GEMINI_API_KEY'])

def extract_resume(raw_text: str, max_retries: int = 1):

    if not raw_text.strip():
        raise ValueError("Resume text is empty")

    for attempt in range(max_retries + 1):

        try:

            resp = client.models.generate_content(
                model='gemini-2.5-flash',
                contents=f"""
Extract a Resume JSON from this text.
Return ONLY JSON, no markdown.

{raw_text}
""",
                config={
                    'response_mime_type': 'application/json',
                    'response_schema': Resume.model_json_schema(),
                },
            )

            return Resume.model_validate_json(resp.text)

        except ValidationError as e:

            if attempt == max_retries:
                raise

            fix_prompt = (
                f"Fix this JSON to match schema. "
                f"Errors: {e}. "
                f"Original: {resp.text}"
            )

            resp = client.models.generate_content(
                model='gemini-2.5-flash',
                contents=fix_prompt,
                config={
                    'response_mime_type': 'application/json',
                    'response_schema': Resume.model_json_schema(),
                },
            )

            return Resume.model_validate_json(resp.text)

In [36]:
import os
print(os.listdir())

['.config', 'Resume.txt', '.ipynb_checkpoints', 'Resume1.txt', 'Resume2.txt', 'sample_data']


In [38]:
resume_files = [
    'Resume.txt',
    'Resume1.txt',
    'Resume2.txt'
]

resumes = []

for file in resume_files:

    try:
        with open(file, 'r', encoding='utf-8') as f:
            resumes.append(f.read())

    except UnicodeDecodeError:

        try:
            with open(file, 'r', encoding='latin-1') as f:
                resumes.append(f.read())

            print(f"{file}: loaded using latin-1 encoding")

        except Exception as e:
            print(f"Failed to read {file}: {e}")

print(f"Loaded {len(resumes)} resumes")

Resume.txt: loaded using latin-1 encoding
Loaded 3 resumes


In [39]:
results = []

for i, r in enumerate(resumes):

    try:

        parsed = extract_resume(r)

        results.append(parsed)

        print(
            f"Resume {i+1}: "
            f"{parsed.name} — "
            f"{len(parsed.skills)} skills, "
            f"{parsed.experience_years} years exp"
        )

    except Exception as e:

        print(
            f"Resume {i+1}: FAILED — "
            f"{type(e).__name__}: {str(e)[:200]}"
        )

Resume 1:  — 0 skills, 0.0 years exp
Resume 2:  — 9 skills, 0.0 years exp
Resume 3: Candidate Name — 14 skills, 0.0 years exp


In [40]:
for i, resume in enumerate(results):

    print(f"\n===== Resume {i+1} =====")

    print(
        resume.model_dump_json(
            indent=2
        )
    )


===== Resume 1 =====
{
  "name": "",
  "email": "",
  "phone": null,
  "education": [],
  "skills": [],
  "projects": [],
  "experience_years": 0.0
}

===== Resume 2 =====
{
  "name": "",
  "email": "",
  "phone": null,
  "education": [
    {
      "degree": "B.Tech in Computer Science and Engineering",
      "institution": "Aditya College of Engineering and Technology, Surampalem",
      "year": 2023
    },
    {
      "degree": "Class XII",
      "institution": "Sri Chaitanya College, Kakinada",
      "year": 2023
    }
  ],
  "skills": [
    "C++",
    "C",
    "Java",
    "Python",
    "HTML",
    "CSS",
    "SQL",
    "Data Structures and Algorithms",
    "Leadership and TeamWork"
  ],
  "projects": [
    "Tic-Tac-Toe Game with AI",
    "Huffman File Compression Tool"
  ],
  "experience_years": 0.0
}

===== Resume 3 =====
{
  "name": "Candidate Name",
  "email": "candidate@example.com",
  "phone": null,
  "education": [
    {
      "degree": "B.Tech in Computer Science and Engine

In [41]:
try:

    bad = extract_resume('')

    print("Unexpected success")

except Exception as e:

    print(
        "Caught gracefully:",
        type(e).__name__
    )

    print(
        "Message:",
        str(e)
    )

Caught gracefully: ValueError
Message: Resume text is empty


In [42]:
import json

all_resumes = [
    r.model_dump()
    for r in results
]

with open(
    "parsed_resumes.json",
    "w"
) as f:

    json.dump(
        all_resumes,
        f,
        indent=2
    )

print("Saved to parsed_resumes.json")

Saved to parsed_resumes.json
